In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/sakshibhosale1904/construction-delay-prediction-dataset/delay_prediction.csv


In [2]:
import pandas as pd

file_path = "/kaggle/input/datasets/sakshibhosale1904/construction-delay-prediction-dataset/delay_prediction.csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (1000, 6)

Columns:
['Project_ID', 'Weather', 'Workers', 'Budget', 'Progress', 'Delay']

First 5 Rows:


,Project_ID,Weather,Workers,Budget,Progress,Delay
0,P0001,Cloudy,109,25080668,84,No
1,P0002,Rain,105,29236617,96,No
2,P0003,Sunny,76,10644034,47,Yes
3,P0004,Rain,73,45294222,87,No
4,P0005,Sunny,37,11716877,43,Yes


In [3]:
print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


Data Types:
Project_ID    object
Weather       object
Workers        int64
Budget         int64
Progress       int64
Delay         object
dtype: object

Missing Values:
Project_ID    0
Weather       0
Workers       0
Budget        0
Progress      0
Delay         0
dtype: int64

Duplicate Rows: 0


In [4]:
print("Delay Distribution:")
print(df["Delay"].value_counts())

print("\nDelay Distribution (%):")
print((df["Delay"].value_counts(normalize=True) * 100).round(2))

Delay Distribution:
Delay
Yes    705
No     295
Name: count, dtype: int64

Delay Distribution (%):
Delay
Yes    70.5
No     29.5
Name: proportion, dtype: float64


In [5]:
# Create a copy
data = df.copy()

# Remove Project_ID because it is only an identifier
data = data.drop(columns=["Project_ID"])

# Encode Weather using one-hot encoding
data = pd.get_dummies(
    data,
    columns=["Weather"],
    drop_first=True
)

# Encode target
data["Delay"] = data["Delay"].map({
    "No": 0,
    "Yes": 1
})

print("Prepared Dataset:")
display(data.head())

print("\nColumns:")
print(data.columns.tolist())

print("\nData Types:")
print(data.dtypes)

Prepared Dataset:


,Workers,Budget,Progress,Delay,Weather_Rain,Weather_Sunny
0,109,25080668,84,0,False,False
1,105,29236617,96,0,True,False
2,76,10644034,47,1,False,True
3,73,45294222,87,0,True,False
4,37,11716877,43,1,False,True



Columns:
['Workers', 'Budget', 'Progress', 'Delay', 'Weather_Rain', 'Weather_Sunny']

Data Types:
Workers          int64
Budget           int64
Progress         int64
Delay            int64
Weather_Rain      bool
Weather_Sunny     bool
dtype: object


In [6]:
X = data.drop(columns=["Delay"])
y = data["Delay"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

X shape: (1000, 5)
y shape: (1000,)

Features:
['Workers', 'Budget', 'Progress', 'Weather_Rain', 'Weather_Sunny']

Target distribution:
Delay
1    705
0    295
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training set: (800, 5)
Testing set : (200, 5)

Training target distribution:
Delay
1    564
0    236
Name: count, dtype: int64

Testing target distribution:
Delay
1    141
0     59
Name: count, dtype: int64


In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

print("✅ Random Forest model trained successfully!")

✅ Random Forest model trained successfully!


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("TEST SET RESULTS")
print("=" * 40)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")

TEST SET RESULTS
Accuracy : 0.6950
Precision: 0.7632
Recall   : 0.8227
F1 Score : 0.7918
ROC-AUC  : 0.6940


In [10]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 50)

for metric in scoring:
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric.upper():10s}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

5-FOLD CROSS-VALIDATION RESULTS
ACCURACY  : 0.6887 ± 0.0392
PRECISION : 0.7472 ± 0.0207
RECALL    : 0.8441 ± 0.0419
F1        : 0.7924 ± 0.0276
ROC_AUC   : 0.7093 ± 0.0353


In [11]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

print("✅ Gradient Boosting model trained successfully!")

✅ Gradient Boosting model trained successfully!


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

gb_pred = gb_model.predict(X_test)
gb_prob = gb_model.predict_proba(X_test)[:, 1]

print("GRADIENT BOOSTING — TEST SET RESULTS")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, gb_pred):.4f}")
print(f"Precision: {precision_score(y_test, gb_pred):.4f}")
print(f"Recall   : {recall_score(y_test, gb_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, gb_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, gb_prob):.4f}")

GRADIENT BOOSTING — TEST SET RESULTS
Accuracy : 0.7200
Precision: 0.7707
Recall   : 0.8582
F1 Score : 0.8121
ROC-AUC  : 0.7331


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

gb_cv_results = cross_validate(
    gb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("GRADIENT BOOSTING — 5-FOLD CROSS-VALIDATION")
print("=" * 55)

for metric in scoring:
    scores = gb_cv_results[f"test_{metric}"]

    print(
        f"{metric.upper():10s}: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

GRADIENT BOOSTING — 5-FOLD CROSS-VALIDATION
ACCURACY  : 0.7050 ± 0.0281
PRECISION : 0.7564 ± 0.0150
RECALL    : 0.8583 ± 0.0381
F1        : 0.8037 ± 0.0208
ROC_AUC   : 0.7234 ± 0.0216


In [14]:
import joblib

joblib.dump(gb_model, "delay_prediction_model.joblib")

print("✅ Final Delay Prediction model saved successfully!")

✅ Final Delay Prediction model saved successfully!


In [15]:
import json

feature_info = {
    "features": list(X.columns),
    "target": "Delay",
    "target_mapping": {
        "No": 0,
        "Yes": 1
    },
    "model_type": type(gb_model).__name__
}

with open("delay_feature_info.json", "w") as f:
    json.dump(feature_info, f, indent=4)

print("✅ Feature information saved successfully!")

✅ Feature information saved successfully!


In [16]:
loaded_gb_model = joblib.load("delay_prediction_model.joblib")

loaded_predictions = loaded_gb_model.predict(X_test)

print("✅ Model loaded successfully!")
print("Number of predictions:", len(loaded_predictions))
print("First 10 predictions:", loaded_predictions[:10])

✅ Model loaded successfully!
Number of predictions: 200
First 10 predictions: [0 1 1 1 1 1 1 1 1 1]
